<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week1_fundamentals/Week1_Lesson2_Workshop.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠 IB CS — Неделя 1 · Урок 2 (Семинар)
## Workshop "Data Integrity" — практика по A4.2.1
### A4.2 Data Preprocessing (HL — Higher Level)

> 📘 Syllabus: **A4.2.1 — *Describe* the significance of data cleaning.**
>
> ⏱ Длительность: 2 академических часа.
> 💻 Платформа: Google Colab (или локальный Jupyter).
> 🎯 Цель: пройти **полный цикл предобработки** руками, чтобы понимать, что и зачем.

---

### ⚠️ ВАЖНО до старта семинара:

1. Откройте этот ноутбук в **Google Colab** (`File → Open → GitHub` или загрузите).
2. Запускайте каждую ячейку **сами**, не просто читайте код.
3. Когда я даю задание `🎯 TRY IT`, остановитесь и сделайте.
4. Записывайте свои IB-style ответы на маркированные вопросы — они появятся в ДЗ.

---

### 📋 План семинара:

| Часть | Тема | Время |
|---|---|---|
| 1 | Загрузка и первичный осмотр (EDA) | 15 мин |
| 2 | Обработка пропусков (Imputation) | 25 мин |
| 3 | Обнаружение выбросов: Z-score и IQR | 30 мин |
| 4 | Нормализация vs Стандартизация | 25 мин |
| 5 | До/После: финальная визуализация | 15 мин |
| 6 | Запись IB-style выводов | 10 мин |


## Часть 1 · Загрузка и первичный осмотр данных

Создадим **синтетический "грязный"** датасет о студентах — близко к тому, что бывает на IA / в Section B.


In [ ]:
# Импортируем библиотеки. Это стандарт для любой ML-задачи в IB.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
sns.set_style('whitegrid')

# Создаём "грязный" датасет — 200 студентов
n = 200
data = {
    'student_id':    range(1, n+1),
    'age':           np.random.normal(17, 1.5, n).round(0),
    'hours_study':   np.random.normal(15, 5, n).round(1),
    'attendance_%':  np.random.normal(85, 10, n).round(1),
    'sleep_hours':   np.random.normal(7, 1.2, n).round(1),
    'final_score':   np.random.normal(75, 12, n).round(0)
}
df = pd.DataFrame(data)

# === Намеренно "пачкаем" данные ===
# 1. Пропуски (NaN)
df.loc[np.random.choice(df.index, 15, replace=False), 'hours_study'] = np.nan
df.loc[np.random.choice(df.index, 10, replace=False), 'sleep_hours'] = np.nan
df.loc[np.random.choice(df.index, 8,  replace=False), 'attendance_%'] = np.nan

# 2. Выбросы (нереалистичные значения)
df.loc[5, 'hours_study']  = 80      # 80 часов в неделю — нереалистично
df.loc[12, 'sleep_hours'] = 0.5     # 0.5 часа сна
df.loc[25, 'final_score'] = 150     # max 100
df.loc[40, 'age']         = 50      # учитель?
df.loc[77, 'attendance_%'] = -10    # отрицательная посещаемость

# 3. Дубликаты
df = pd.concat([df, df.iloc[[3, 7]]], ignore_index=True)

print(f"Размер датасета: {df.shape}")
df.head(10)

### 🔍 EDA — Exploratory Data Analysis

> 💎 **СЕКРЕТ #1:** в IB IA (Internal Assessment) экзаменаторы хотят видеть EDA **в самом начале**. Это показывает, что вы понимаете данные ДО построения модели.


In [ ]:
# 1. Базовая информация
print("=== df.info() ===")
df.info()
print("\n=== Пропуски по колонкам ===")
print(df.isnull().sum())
print(f"\nВсего строк: {len(df)}")
print(f"Дубликатов: {df.duplicated().sum()}")

In [ ]:
# 2. Описательная статистика — главный инструмент в IB IA
df.describe()

> 🎯 **TRY IT #1:** Посмотрите на `describe()`. Что вы заметили **подозрительного**?
>
> Подсказки:
> - `age` — какое max значение?
> - `attendance_%` — какое min?
> - `hours_study` — какое max?
> - `final_score` — какое max?
>
> 📝 Запишите 3–4 аномалии. Это и есть то, что нужно "чистить".


In [ ]:
# 3. Визуальный осмотр — boxplots для всех числовых колонок
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, ['age', 'hours_study', 'attendance_%', 'sleep_hours', 'final_score']):
    sns.boxplot(y=df[col], ax=ax, color='lightblue')
    ax.set_title(col, fontsize=11, fontweight='bold')
plt.suptitle('Boxplots ДО очистки — точки за "усами" = potential outliers', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

### 📚 IB определения (из Baumgarten, A4.2.1)

> **Outlier:** *a data point that deviates from the typical pattern of values in a data set, indicating a possible unusual or erroneous value that should be discounted.*

**7 шагов очистки данных (по учебнику Baumgarten):**

1. **Handling outliers** — обработка выбросов (IQR, Z-score)
2. **Removing duplicate data** — удаление дубликатов
3. **Identifying incorrect data** — валидация (например, отрицательная посещаемость)
4. **Filtering irrelevant data** — удаление нерелевантных столбцов
5. **Transform improperly formatted data** — унификация форматов
6. **Missing data** — заполнение пропусков (imputation)
7. **Normalization and standardization** — масштабирование

Сегодня мы пройдём пункты **1, 2, 3, 6, 7** на практике.


## Часть 2 · Обработка пропусков (Imputation)

> 📘 **IB Top Tip из Baumgarten:**
> *"Inputting missing data will improve model accuracy through providing a complete data set, but it can introduce bias if the resulting data set does not match actual data distribution."*

**3 главных стратегии:**

| Метод | Когда использовать | Минус |
|---|---|---|
| **Deletion** (удалить строки с NaN) | пропусков очень мало (<5%) | теряете данные |
| **Mean / Median imputation** | пропуски случайны (MCAR) | искажает распределение, занижает дисперсию |
| **Mode imputation** | для категориальных переменных | то же |
| **KNN / Regression imputation** | пропусков много, нужна точность | сложнее, дольше |


In [ ]:
# Стратегия А: посмотрим количество пропусков
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'Пропусков': missing, '%': missing_pct})

In [ ]:
# Стратегия Б: Mean imputation для hours_study
# Сохраним копию до для сравнения
df_before = df.copy()

# ══════════ 🎯 ПРОПУСК 1: Imputation ══════════
# TODO: заполните пропуски в hours_study СРЕДНИМ значением
# Подсказка: методы .mean() и .fillna()
mean_hours = ...
df['hours_study'] = ...
print(f"Заполнено средним: {mean_hours:.2f} часов")

# TODO: заполните пропуски в sleep_hours МЕДИАНОЙ
# (почему median, а не mean? — вспомните про выброс 0.5 часа сна!)
median_sleep = ...
df['sleep_hours'] = ...
print(f"Заполнено медианой: {median_sleep:.2f} часов")

# TODO: заполните пропуски в attendance_% медианой (там есть выброс −10)
median_att = ...
df['attendance_%'] = ...
print(f"Заполнено медианой: {median_att:.2f}%")
# ══════════ конец пропуска 1 ══════════

print("\n=== Пропуски ПОСЛЕ ===")
print(df.isnull().sum())

### 🎯 TRY IT #2: Почему для `hours_study` я выбрал **mean**, а для `sleep_hours` — **median**?

> 💡 Подумайте о различии:
> - **Mean** чувствителен к выбросам.
> - **Median** устойчив к выбросам.
>
> Какое значение есть в `sleep_hours` из аномалий? (вернитесь к коду пачкания данных) → именно поэтому median.

> 💎 **СЕКРЕТ #2 для IB:** на экзамене всегда **обосновывайте** выбор метода. Просто "I used mean" = 0 баллов. *"I used median because the data contains outliers (e.g., sleep_hours = 0.5), which would skew the mean"* = full marks.


In [ ]:
# Визуализация: распределение ДО и ПОСЛЕ заполнения
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df_before['hours_study'].dropna(), bins=20, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(mean_hours, color='red', linestyle='--', label=f'Mean = {mean_hours:.1f}')
axes[0].set_title('hours_study — ДО (с пропусками)')
axes[0].set_xlabel('Часы'); axes[0].legend()

axes[1].hist(df['hours_study'], bins=20, color='green', alpha=0.7, edgecolor='black')
axes[1].axvline(mean_hours, color='red', linestyle='--', label=f'Mean = {mean_hours:.1f}')
axes[1].set_title('hours_study — ПОСЛЕ mean imputation')
axes[1].set_xlabel('Часы'); axes[1].legend()

plt.tight_layout(); plt.show()

> 🔬 **Наблюдение:** заметили, что пик у среднего стал выше? Это **bias** — mean imputation **искажает** распределение (искусственно увеличивает плотность около среднего).
>
> Именно об этом предупреждает Baumgarten. На экзамене это можно использовать как аргумент в `Discuss` вопросах.


In [ ]:
# Часть 2.5 · Удаление дубликатов
print(f"Дубликатов до: {df.duplicated().sum()}")

# ══════════ 🎯 ПРОПУСК 2: Дубликаты ══════════
# TODO: удалите дубликаты и сбросьте индекс
# Подсказка: .drop_duplicates() и .reset_index(drop=True)
df = ...
# ══════════ конец пропуска 2 ══════════

print(f"Дубликатов после: {df.duplicated().sum()}")
print(f"Размер датасета: {df.shape}")

## Часть 3 · Обнаружение выбросов: Z-score и IQR

> **Outlier** (по syllabus): a data point that deviates from the typical pattern of values...
>
> Два главных метода (в IB Baumgarten страница 22):

### Метод 1: **Z-score** (нормализованное отклонение)

$$Z = \frac{x - \mu}{\sigma}$$

- $\mu$ — среднее, $\sigma$ — стандартное отклонение.
- **Правило:** если $|Z| > 3$ — это outlier (правило 3 сигм для нормального распределения).

### Метод 2: **IQR** (Interquartile Range) — БОЛЕЕ УСТОЙЧИВ к выбросам

$$IQR = Q_3 - Q_1$$

- Outlier, если $x < Q_1 - 1.5 \cdot IQR$ или $x > Q_3 + 1.5 \cdot IQR$.

> 💎 **СЕКРЕТ #3:** Z-score предполагает **нормальное** распределение. Если данные скошенные → **используйте IQR**. На экзамене это бонусный балл за глубину понимания.


In [ ]:
# === Z-score метод ===
def find_outliers_zscore(series, threshold=3):
    # ══════════ 🎯 ПРОПУСК 3a: Z-score ══════════
    # TODO: вычислите |Z| для каждой точки
    # Подсказка: stats.zscore() и np.abs()
    z = ...
    # ══════════ конец пропуска 3a ══════════
    return series[z > threshold]

# === IQR метод ===
def find_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    # ══════════ 🎯 ПРОПУСК 3b: IQR ══════════
    # TODO: вычислите IQR и границы выбросов
    # Формула: lower = Q1 − 1.5·IQR, upper = Q3 + 1.5·IQR
    IQR = ...
    lower = ...
    upper = ...
    # ══════════ конец пропуска 3b ══════════
    return series[(series < lower) | (series > upper)], lower, upper

# Проверим колонку final_score (там у нас 150 — невозможный балл)
print("=== final_score ===")
print("\nZ-score outliers:")
print(find_outliers_zscore(df['final_score']))

outliers, low, high = find_outliers_iqr(df['final_score'])
print(f"\nIQR outliers (диапазон допустимого: {low:.1f} — {high:.1f}):")
print(outliers)

In [ ]:
# Визуализация выбросов на всех колонках
numeric_cols = ['age', 'hours_study', 'attendance_%', 'sleep_hours', 'final_score']
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i, col in enumerate(numeric_cols):
    # Сверху — boxplot
    sns.boxplot(y=df[col], ax=axes[0, i], color='salmon')
    axes[0, i].set_title(f'{col}\n(Boxplot — IQR)', fontsize=11)

    # Снизу — histogram с границами Z-score
    axes[1, i].hist(df[col], bins=20, color='lightblue', edgecolor='black')
    mu, sigma = df[col].mean(), df[col].std()
    axes[1, i].axvline(mu - 3*sigma, color='red', linestyle='--', label='−3σ')
    axes[1, i].axvline(mu + 3*sigma, color='red', linestyle='--', label='+3σ')
    axes[1, i].axvline(mu, color='green', label='mean')
    axes[1, i].set_title(f'{col}\n(Histogram + Z-score)', fontsize=11)
    axes[1, i].legend(fontsize=8)

plt.tight_layout(); plt.show()

### 🎯 TRY IT #3: Что делать с выбросами?

По учебнику Baumgarten — **три варианта**:
1. **Capping** (обрезать до границы): `final_score = 150 → 100`
2. **Transformation** (логарифм, sqrt)
3. **Removal** (удалить запись)

> 💡 **Выбор зависит от контекста:**
> - `final_score = 150` — это явно **ошибка ввода** (max = 100). → **Capping** до 100, либо удалить.
> - `attendance_% = −10` — невозможно. → **Removal** или замена медианой.
> - `age = 50` — может быть и валидно (взрослый студент?), но в школе — скорее ошибка. **Контекст важен!**

> 💎 **СЕКРЕТ #4:** на экзамене *"Describe how outliers could affect the performance of the linear regression model"* — ответ: *outliers disproportionately influence the slope of the regression line because least-squares minimization is sensitive to large residuals; this can lead to poor predictions for typical data points.*


In [ ]:
# ══════════ 🎯 ПРОПУСК 4: Обработка выбросов ══════════
# TODO: примените capping для final_score и attendance_% — оба в диапазон [0, 100]
# Подсказка: метод .clip(min, max)
df['final_score'] = ...
df['attendance_%'] = ...

# TODO: для age — оставьте только строки с возрастом от 14 до 19 включительно
# Подсказка: булева маска df[(условие1) & (условие2)] + .reset_index(drop=True)
df = ...
# ══════════ конец пропуска 4 ══════════

# Для sleep_hours = 0.5 → cap минимум до 3 (нереалистично меньше)
df['sleep_hours'] = df['sleep_hours'].clip(3, 12)

# Для hours_study = 80 → cap максимум до 60
df['hours_study'] = df['hours_study'].clip(0, 60)

print("После обработки выбросов:")
df.describe()

## Часть 4 · Нормализация vs Стандартизация (A4.2.1, конец)

> 📘 **Baumgarten:**
> *"Normalization can be used to rescale input data to a range of [0,1] or [−1,1]. Standardization can be used to transform the input data to have a mean of 0 and standard deviation of 1 (Gaussian distribution)."*

### 🔑 Разница (часто путают!):

| | **Normalization (Min-Max)** | **Standardization (Z-score)** |
|---|---|---|
| Формула | $x' = \frac{x - x_{min}}{x_{max} - x_{min}}$ | $x' = \frac{x - \mu}{\sigma}$ |
| Диапазон | [0, 1] | mean=0, std=1 (любой диапазон) |
| Когда | KNN, нейросети, изображения | алгоритмы, предполагающие нормальное распределение (linear regression, logistic regression) |
| Чувствительность к выбросам | **высокая** (выброс растягивает шкалу) | средняя |

> 🚨 **Common mistake (Baumgarten, прямая цитата):**
> *"Ignoring the important role that normalization and standardization play. Apply these transformations consistently across all data used in the model."*

> 💎 **СЕКРЕТ #5:** на экзамене *"Why do KNN algorithms need normalization?"* — ответ: KNN считает **расстояния**. Если одна фича — `salary` (диапазон 0–1 000 000), а другая — `age` (0–100), то salary будет **полностью доминировать** в расчётах расстояний. Нормализация уравнивает их вклад.


In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

cols_to_scale = ['hours_study', 'attendance_%', 'sleep_hours', 'final_score']

# ══════════ 🎯 ПРОПУСК 5: Нормализация vs Стандартизация ══════════
# TODO: примените MinMaxScaler к колонкам cols_to_scale
# Подсказка: метод .fit_transform()
mm = MinMaxScaler()
df_norm = df.copy()
df_norm[cols_to_scale] = ...

# TODO: примените StandardScaler к тем же колонкам
ss = StandardScaler()
df_std = df.copy()
df_std[cols_to_scale] = ...
# ══════════ конец пропуска 5 ══════════

print("=== ОРИГИНАЛ ===")
print(df[cols_to_scale].describe().loc[['mean', 'std', 'min', 'max']].round(2))
print("\n=== NORMALIZED (Min-Max, должно быть [0,1]) ===")
print(df_norm[cols_to_scale].describe().loc[['mean', 'std', 'min', 'max']].round(2))
print("\n=== STANDARDIZED (mean=0, std=1) ===")
print(df_std[cols_to_scale].describe().loc[['mean', 'std', 'min', 'max']].round(2))

In [ ]:
# Визуализация: одна и та же колонка в 3 форматах
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['final_score'], bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Оригинал\n(0–100)', fontweight='bold')
axes[0].set_xlabel('final_score')

axes[1].hist(df_norm['final_score'], bins=20, color='green', edgecolor='black')
axes[1].set_title('Normalized\n[0, 1]', fontweight='bold')
axes[1].set_xlabel('final_score (norm)')

axes[2].hist(df_std['final_score'], bins=20, color='orange', edgecolor='black')
axes[2].set_title('Standardized\n(mean=0, std=1)', fontweight='bold')
axes[2].set_xlabel('final_score (std)')

plt.tight_layout(); plt.show()

## Часть 5 · Финальное сравнение «ДО ↔ ПОСЛЕ»

Покажем, **что именно** улучшилось благодаря очистке.


In [ ]:
# Сравнительные boxplots
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i, col in enumerate(['age', 'hours_study', 'attendance_%', 'sleep_hours', 'final_score']):
    sns.boxplot(y=df_before[col].dropna(), ax=axes[0, i], color='salmon')
    axes[0, i].set_title(f'ДО — {col}', fontweight='bold')

    sns.boxplot(y=df[col], ax=axes[1, i], color='lightgreen')
    axes[1, i].set_title(f'ПОСЛЕ — {col}', fontweight='bold')

plt.suptitle('Сравнение ДО ↔ ПОСЛЕ очистки', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Корреляционная матрица — теперь без шума она будет читаться чище
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(df_before[['age','hours_study','attendance_%','sleep_hours','final_score']].corr(),
            annot=True, cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('Корреляции ДО очистки')

sns.heatmap(df[['age','hours_study','attendance_%','sleep_hours','final_score']].corr(),
            annot=True, cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Корреляции ПОСЛЕ очистки')

plt.tight_layout(); plt.show()

### 🎯 TRY IT #4: Сравните корреляции «До» и «После».

Изменились ли значения? Какая фича теперь сильнее коррелирует с `final_score`?

> 💡 Это **критично** для следующего этапа — **feature selection** (A4.2.2). Без чистых данных корреляции врут.


## Часть 6 · Запись IB-style выводов

> 💎 **СЕКРЕТ #6:** в IB IA / Section B вам нужно **обосновать каждый шаг** предобработки. Используйте структуру:
>
> **Шаг → Метод → Обоснование → Эффект на модель**

### Образцовый IB-style вывод (заполните для своего отчёта):

```
1. Missing values:
   - В колонке `hours_study` (15 пропусков, 7.5%) применена mean imputation,
     так как распределение близко к нормальному и пропуски случайны.
   - В колонке `sleep_hours` (10 пропусков) применена MEDIAN imputation,
     так как присутствует выброс (0.5), который сместил бы среднее значение.

2. Outliers:
   - В `final_score` использован capping до 100, так как значение 150
     невозможно (максимум — 100 баллов).
   - В `attendance_%` исправлено отрицательное значение через capping в [0, 100].

3. Normalization:
   - Применён MinMaxScaler для всех числовых фич перед использованием в KNN,
     так как KNN основан на евклидовом расстоянии и чувствителен к разным шкалам.

Effect on model:
- Cleaning reduced standard deviation of `final_score` from 13.2 to 11.8,
  removing artificial inflation caused by the outlier.
- Normalization ensured all features contribute equally to distance calculations,
  preventing dominance of `attendance_%` (range 0–100) over `sleep_hours` (range 3–12).
```


## ✅ Что мы сделали за семинар

- [x] Загрузили "грязный" датасет и провели EDA
- [x] Обработали **пропуски** (mean / median imputation) — A4.2.1 ✓
- [x] Нашли **выбросы** двумя методами (Z-score, IQR) и обработали (capping, removal) — A4.2.1 ✓
- [x] Удалили **дубликаты** — A4.2.1 ✓
- [x] Применили **нормализацию** и **стандартизацию** — A4.2.1 ✓
- [x] Визуализировали ДО ↔ ПОСЛЕ — техника для IA
- [x] Научились писать **IB-style обоснование** каждого шага

---

### 🏠 Что в ДЗ (см. `Week1_HW2_Practice.ipynb`)

> Реальный "грязный" датасет цен на недвижимость. Полный цикл предобработки + IB-style отчёт.

---

### 💎 Финальные секреты семинара

1. **Всегда** обосновывайте выбор метода — это даёт половину баллов.
2. **Всегда** показывайте ДО ↔ ПОСЛЕ — графики дают визуальное доказательство.
3. **Cap before normalize** — выбросы искажают min-max scaling.
4. **Median > Mean** при наличии выбросов.
5. На вопрос *"Describe significance of data cleaning"* (типичный 4-балльный вопрос) — структура:
   - Что улучшает (accuracy, generalization)
   - Что предотвращает (bias, overfitting)
   - Пример конкретный
   - Trade-off (потеря данных при удалении)
